# DAY 161 - Question & Answering (Q&A).
@A.IPYNB

**We** are combining Project A (Semantic Search) and Project B (Question Answering) into a single, intelligent system known as Extractive RAG (Retrieval-Augmented Generation).

### This is essentially building your own "Mini-Google."

We ask: "Who walked on the moon?"

Retriever (SBERT): Scans 1,000 documents to find the one paragraph about Apollo 11.

Reader (QA): Reads that paragraph and extracts "Neil Armstrong."

## The Full Pipeline: Extractive RAG

### How it Works
We are building a system that can answer questions based on a specific Knowledge Base (e.g., your company manuals, history books, or technical docs).

1.  **The Retriever (The Librarian):**
    * Uses **SBERT** to convert your Query and all Documents into vectors.
    * Finds the document with the highest *Cosine Similarity* (Semantic Match).
    * *Goal:* Narrow down the haystack to finding the one needle.

2.  **The Reader (The Analyst):**
    * Uses **DistilBERT-QA** to read the retrieved document.
    * Extracts the exact answer span.
    * *Goal:* Give you the specific fact, not the whole page.

In [1]:
# @title 1. Dependencies
!pip install -q sentence-transformers transformers torch

import torch
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

print("Libraries installed. Building the brain...")

Libraries installed. Building the brain...


In [2]:
# @title 2. The Knowledge Base
# In a real app, this would be a PDF or a Database.
# Here, we simulate a library with 5 distinct topics.

knowledge_base = [
    # Topic 1: Space
    """The Apollo 11 mission was the first spaceflight that landed the first two people on the Moon.
    Commander Neil Armstrong and lunar module pilot Buzz Aldrin, both American, landed the Apollo Lunar Module Eagle on July 20, 1969.""",

    # Topic 2: Biology
    """Mitochondria are known as the powerhouses of the cell. They are organelles that act like a digestive system
    which takes in nutrients, breaks them down, and creates energy rich molecules for the cell.""",

    # Topic 3: History
    """The Great Wall of China is a series of fortifications that were built across the historical northern borders of ancient Chinese states
    and Imperial China as protection against various nomadic groups from the Eurasian Steppe.""",

    # Topic 4: Technology
    """Python is a high-level, general-purpose programming language. Its design philosophy emphasizes code readability
    with the use of significant indentation. Python is dynamically typed and garbage-collected.""",

    # Topic 5: Geography
    """The Amazon River in South America is the largest river by discharge volume of water in the world,
    and the second in length. It flows through Brazil, Peru, and Colombia."""
]

print(f"Knowledge Base Loaded: {len(knowledge_base)} documents.")

Knowledge Base Loaded: 5 documents.


In [3]:
# @title 4. The Reader
# This model will never see the whole library. It only sees what the Retriever gives it.

reader = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

print("Reader is ready.")

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Reader is ready.


In [5]:
# @title 5. The RAG Pipeline
# Connect Retriever -> Reader

# This model converts text into vectors for comparison.
retriever = SentenceTransformer('all-MiniLM-L6-v2')
print("Retriever initialized.")

# Pre-compute embeddings for the knowledge base
corpus_embeddings = retriever.encode(knowledge_base, convert_to_tensor=True)
print(f"Knowledge base embedded: {corpus_embeddings.shape[0]} documents.")

def retrieve_doc(query):
    # This is to Encode the query
    query_embedding = retriever.encode(query, convert_to_tensor=True)

    # This is to Compute cosine-similarity scores between the query and all corpus embeddings
    cosine_scores = util.cos_sim(query_embedding, corpus_embeddings)[0]

    # This is to Find the best match
    best_doc_idx = torch.argmax(cosine_scores).item()
    best_doc_score = cosine_scores[best_doc_idx].item()

    return knowledge_base[best_doc_idx], best_doc_score

def ask_the_oracle(query):
    print(f"Query: {query}")

    # Step 1: This is Retrieves
    context, score = retrieve_doc(query)
    print(f"--> Retrieved Doc (Score: {score:.2f}):\n    ...{context[:100]}...")

    # Step 2: This is to Extracts Answer
    result = reader(question=query, context=context)
    print(f"--> Answer: {result['answer']}")
    print(f"--> Confidence: {result['score']:.2f}")
    print("-" * 50)

# TEST DRIVE
ask_the_oracle("Who landed on the moon?")
ask_the_oracle("What is the powerhouse of the cell?")
ask_the_oracle("Which language uses indentation?")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retriever initialized.
Knowledge base embedded: 5 documents.
Query: Who landed on the moon?
--> Retrieved Doc (Score: 0.67):
    ...The Apollo 11 mission was the first spaceflight that landed the first two people on the Moon.
    Co...
--> Answer: Commander Neil Armstrong and lunar module pilot Buzz Aldrin
--> Confidence: 0.40
--------------------------------------------------
Query: What is the powerhouse of the cell?
--> Retrieved Doc (Score: 0.66):
    ...Mitochondria are known as the powerhouses of the cell. They are organelles that act like a digestive...
--> Answer: Mitochondria
--> Confidence: 1.00
--------------------------------------------------
Query: Which language uses indentation?
--> Retrieved Doc (Score: 0.41):
    ...Python is a high-level, general-purpose programming language. Its design philosophy emphasizes code ...
--> Answer: Python
--> Confidence: 0.93
--------------------------------------------------


### The Strategic Win

We have built a fully functional **Open-Domain Question Answering System**.

1.  **Scalability:** We can add 1,000,000 more documents to `knowledge_base`. The **Retriever** will still find the right one in milliseconds.
2.  **Precision:** The **Reader** doesn't hallucinate. It only extracts answers that physically exist in the retrieved text.
3.  **Efficiency:** Instead of feeding a 500-page book into ChatGPT (which is expensive), We only feed the single relevant paragraph.

**This is the architecture behind:**
* Corporate Search Engines ("How do I file expenses?")
* Legal Discovery Tools ("Find cases about negligence.")
* Medical Assistants ("What are the side effects of Aspirin?")